In [1]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [2]:
%pip install -q ultralytics pyyaml pandas

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.9/46.9 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 36.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 96.0/96.0 kB 10.0 MB/s eta 0:00:00


In [3]:
!pip install -q -U ultralytics

In [4]:
import torch
import ultralytics

print("PyTorch:", torch.__version__)
print("Ultralytics:", ultralytics.__version__)
print("CUDA:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("Cảnh báo: nên bật T4 GPU trước khi train.")

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.
PyTorch: 2.11.0+cu130
Ultralytics: 8.4.173
CUDA: True
GPU: Tesla T4


In [5]:
from pathlib import Path
import yaml

# Chọn một trong ba:
DATASET_VARIANT = "clahe"

REPO_DIR = Path(
    "/content/Low-Light-Image-Enhancement-and-Downstream-Recognition"
)

DATASET_PATHS = {
    "dark": (
        REPO_DIR
        / "Dataset"
        / "exdark_yolo_dark"
    ),

    "clahe": Path(
        "/content/drive/MyDrive/Project18_Datasets/exdark_yolo_clahe"
    ),

    "zerodcepp": Path(
        "/content/drive/MyDrive/Project18_Datasets/exdark_yolo_zerodcepp"
    ),
}

DATASET_ROOT = DATASET_PATHS[DATASET_VARIANT]

RESULTS_ROOT = Path(
    "/content/drive/MyDrive/Project18_Results"
)

RESULTS_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

assert DATASET_ROOT.exists(), (
    f"Không tìm thấy dataset: {DATASET_ROOT}"
)

source_yaml = DATASET_ROOT / "data.yaml"

assert source_yaml.exists(), (
    f"Không tìm thấy data.yaml: {source_yaml}"
)

with open(source_yaml, "r", encoding="utf-8") as file:
    data_config = yaml.safe_load(file)

# Đảm bảo YOLO luôn dùng đúng đường dẫn trên Colab
data_config["path"] = DATASET_ROOT.as_posix()
data_config["train"] = "train/images"
data_config["val"] = "valid/images"
data_config["test"] = "test/images"

if "names" in data_config:
    data_config["nc"] = len(data_config["names"])

COLAB_DATA_YAML = (
    DATASET_ROOT
    / "data_colab.yaml"
)

with open(
    COLAB_DATA_YAML,
    "w",
    encoding="utf-8",
) as file:
    yaml.safe_dump(
        data_config,
        file,
        sort_keys=False,
        allow_unicode=True,
    )

print("Dataset:", DATASET_VARIANT)
print("Dataset root:", DATASET_ROOT)
print("Data YAML:", COLAB_DATA_YAML)
print("Results:", RESULTS_ROOT)
print()
print(COLAB_DATA_YAML.read_text(encoding="utf-8"))

Dataset: clahe
Dataset root: /content/drive/MyDrive/Project18_Datasets/exdark_yolo_clahe
Data YAML: /content/drive/MyDrive/Project18_Datasets/exdark_yolo_clahe/data_colab.yaml
Results: /content/drive/MyDrive/Project18_Results

path: /content/drive/MyDrive/Project18_Datasets/exdark_yolo_clahe
train: train/images
val: valid/images
test: test/images
names:
  0: Bicycle
  1: Boat
  2: Bottle
  3: Bus
  4: Cat
  5: Cup
  6: Motorbike
  7: People
  8: Table
  9: car
  10: chair
  11: dog
nc: 12



In [6]:
for split in [
    "train",
    "valid",
    "test",
]:
    image_dir = (
        DATASET_ROOT
        / split
        / "images"
    )

    label_dir = (
        DATASET_ROOT
        / split
        / "labels"
    )

    image_count = len([
        path
        for path in image_dir.iterdir()
        if path.suffix.lower() in {
            ".jpg", ".jpeg", ".png",
            ".bmp", ".webp"
        }
    ])

    label_count = len(
        list(label_dir.glob("*.txt"))
    )

    status = (
        "OK"
        if image_count == label_count
        else "LỖI"
    )

    print(
        f"{split:5s}: "
        f"{image_count} ảnh, "
        f"{label_count} labels — "
        f"{status}"
    )

    assert image_count == label_count

train: 5142 ảnh, 5142 labels — OK
valid: 1469 ảnh, 1469 labels — OK
test : 734 ảnh, 734 labels — OK


In [7]:
from ultralytics import YOLO
from datetime import datetime

# Cấu hình train
MODEL_NAME = "yolov8n.pt"
EPOCHS = 40
IMAGE_SIZE = 640
BATCH_SIZE = 16
PATIENCE = 15
SEED = 42

DEVICE = 0 if torch.cuda.is_available() else "cpu"

# Tên riêng giúp không ghi đè kết quả cũ
time_string = datetime.now().strftime(
    "%Y%m%d_%H%M%S"
)

RUN_NAME = (
    f"yolov8n_{DATASET_VARIANT}_"
    f"e{EPOCHS}_{time_string}"
)

print("Model:", MODEL_NAME)
print("Dataset:", DATASET_VARIANT)
print("Run:", RUN_NAME)
print("Device:", DEVICE)

model = YOLO(MODEL_NAME)

train_results = model.train(
    data=str(COLAB_DATA_YAML),

    epochs=EPOCHS,
    imgsz=IMAGE_SIZE,
    batch=BATCH_SIZE,

    device=DEVICE,
    workers=2,

    seed=SEED,
    deterministic=True,

    patience=PATIENCE,
    close_mosaic=10,

    pretrained=True,
    optimizer="auto",

    save=True,
    save_period=10,
    plots=True,

    project=str(RESULTS_ROOT),
    name=RUN_NAME,
    exist_ok=False,
)

RUN_DIR = Path(
    model.trainer.save_dir
)

BEST_WEIGHTS = (
    RUN_DIR
    / "weights"
    / "best.pt"
)

LAST_WEIGHTS = (
    RUN_DIR
    / "weights"
    / "last.pt"
)

print("\nTrain hoàn tất")
print("Thư mục kết quả:", RUN_DIR)
print("Best weights:", BEST_WEIGHTS)
print("Last weights:", LAST_WEIGHTS)
print("Training CSV:", RUN_DIR / "results.csv")

Model: yolov8n.pt
Dataset: clahe
Run: yolov8n_clahe_e40_20261005_064031
Device: 0
Ultralytics 8.4.173 🚀 Python-3.13.15 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/drive/MyDrive/Project18_Datasets/exdark_yolo_clahe/data_colab.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=40, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train,